# clip-grad-norm-pre-step — worked example 2: Verify clip returns pre-clip norm while post-norm is capped

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `clip-grad-norm-pre-step`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`clip_grad_norm_` returns the global gradient norm measured BEFORE any rescaling. After it runs, the *actual* norm of the grads is `min(pre_norm, max_norm)`. This lets you log the raw spike magnitude (the return value) while the optimizer only ever sees the capped gradient.

## Worked solution

**Step 1 - manufacture a large gradient.** We give a single parameter a hand-set `.grad` of all 3.0 across 4 elements, so the true norm is `sqrt(4 * 9) = 6.0`, comfortably above a `max_norm` of 2.0. Setting `.grad` directly lets us know the ground truth without a backward pass.

**Step 2 - call clip and capture the return.** `clip_grad_norm_([p], 2.0)` returns a tensor equal to 6.0 (the pre-clip norm). This is the value you would log.

**Step 3 - measure the post-clip norm independently.** We recompute the norm of `p.grad` after clipping. Because 6.0 > 2.0, every element was multiplied by `2.0 / 6.0`, so the new norm is exactly 2.0. The return value (6.0) and the post-clip norm (2.0) are deliberately different things.

**Why it works:** the rescale factor `max_norm / pre_norm` is applied uniformly, so the resulting norm equals `max_norm` exactly when clipping triggers, while the returned scalar preserves the diagnostic information about how big the spike was.

In [ ]:
import torch.nn.utils as nn_utils

t.manual_seed(0)

p = t.zeros(4, requires_grad=True)
p.grad = t.full((4,), 3.0)

pre = nn_utils.clip_grad_norm_([p], max_norm=2.0)
post = p.grad.norm().item()

print("pre-clip norm:", round(pre.item(), 4))
print("post-clip norm:", round(post, 4))